In [11]:
!uv pip install langchain langchain-community langchain-openai langchain-text-splitters youtube-transcript-api faiss-cpu

Checked 6 packages in 31ms


In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

C:\Users\HP\AppData\Local\Temp\ipykernel_2872\2470843845.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
load_dotenv()

True

In [7]:
yt_api = YouTubeTranscriptApi()
video_id='SfOaZIGJ_gs'
transcript= yt_api.fetch(video_id)
print(transcript)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Music]', start=0.87, duration=1.82), FetchedTranscriptSnippet(text='[Applause]', start=1.58, duration=18.69), FetchedTranscriptSnippet(text='[Music]', start=2.69, duration=17.58), FetchedTranscriptSnippet(text='uh where Nik', start=21.84, duration=4.16), FetchedTranscriptSnippet(text='>> you told him 5 minutes', start=24.8, duration=2.88), FetchedTranscriptSnippet(text='>> like he has 2 minutes not', start=26.0, duration=4.32), FetchedTranscriptSnippet(text='>> yeah 2 minutes Everything looks good.', start=27.68, duration=5.84), FetchedTranscriptSnippet(text='Just the monitor, the main monitor uh', start=30.32, duration=4.16), FetchedTranscriptSnippet(text='went off.', start=33.52, duration=6.199), FetchedTranscriptSnippet(text=">> Everyone who's done can leave.", start=34.48, duration=5.239), FetchedTranscriptSnippet(text='>> Hi, fam.', start=43.84, duration=2.16), FetchedTranscriptSnippet(text='>> Hey, Nicole.', start=44.879

In [8]:
[doc.text for doc in transcript]

['[Music]',
 '[Applause]',
 '[Music]',
 'uh where Nik',
 '>> you told him 5 minutes',
 '>> like he has 2 minutes not',
 '>> yeah 2 minutes Everything looks good.',
 'Just the monitor, the main monitor uh',
 'went off.',
 ">> Everyone who's done can leave.",
 '>> Hi, fam.',
 '>> Hey, Nicole.',
 '>> How are you?',
 ">> I'm good. Sorry I'm late. I got caught",
 'up in getting ready for the launch',
 'tomorrow and lost track of time and',
 'excitement with the final results. But',
 ">> no worries. I'm guessing it must be",
 'really hectic, right?',
 '>> It is a very hectic day.',
 '[Music]',
 '[Music]',
 "I have the model and I've been playing",
 'with it a little bit. How is it',
 "different, Sam? I'm not an expert at",
 'this. So yeah,',
 ">> there's all these ways we can talk",
 "about, you know, it's better at this",
 "metric or it's, you know, can do this",
 'amazing coding demo that the, you know,',
 "GPT4 couldn't. But the thing that has",
 'been most striking for me is in ways',
 '

In [9]:
transcripts_text="".join(doc.text for doc in transcript)

In [10]:
transcripts_text

'[Music][Applause][Music]uh where Nik>> you told him 5 minutes>> like he has 2 minutes not>> yeah 2 minutes Everything looks good.Just the monitor, the main monitor uhwent off.>> Everyone who\'s done can leave.>> Hi, fam.>> Hey, Nicole.>> How are you?>> I\'m good. Sorry I\'m late. I got caughtup in getting ready for the launchtomorrow and lost track of time andexcitement with the final results. But>> no worries. I\'m guessing it must bereally hectic, right?>> It is a very hectic day.[Music][Music]I have the model and I\'ve been playingwith it a little bit. How is itdifferent, Sam? I\'m not an expert atthis. So yeah,>> there\'s all these ways we can talkabout, you know, it\'s better at thismetric or it\'s, you know, can do thisamazing coding demo that the, you know,GPT4 couldn\'t. But the thing that hasbeen most striking for me is in waysthat are both big and small, going backfrom GPT5 to our previous generationmodel is just so painful. It\'s just likeworse at everything. And I\'ve take

In [11]:
## Text Splitting

In [12]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [13]:
chunks = splitter.split_text(transcripts_text)

In [14]:
len(chunks)

86

In [33]:
## Embedding Model

In [15]:
!uv pip install langchain-google-genai

Checked 1 package in 1.36s


In [16]:
import os
print("API Key Loaded:", "GEMINI_API_KEY" in os.environ)

API Key Loaded: True


In [17]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
embeddings = GoogleGenerativeAIEmbeddings(
    model = "gemini-embedding-2-preview",
    task_type = "RETRIEVAL_QUERY"
)

In [50]:
## Vector Store

In [18]:
vector_store=FAISS.from_texts(chunks,embeddings)

In [53]:
vector_store

In [19]:
vector_store.similarity_search(query="Tell me about AI ?")

[Document(id='097089c7-5385-4f7a-b0ca-4af075467ecf', metadata={}, page_content="in this AI realm for India as acountry? What's the opportunity for us?>> As I mentioned earlier, I think Indiamay be our largest market in the worlduh at some point in the not very distantfuture. the excitement,the embrace of AI in India and theability to for Indian people to use AIto just sort of leaprog into the futureand uh invent a totally new and betterway of doing things and the sort of theeconomic benefits that come from thatthe societal benefits if if there is onelarge society in the"),
 Document(id='223e06a7-c377-470f-a5a9-a29ac260bde1', metadata={}, page_content="uh help you write thesoftware for a product much moreefficiently, help you uh handle customersupport, help you write marketing andcommunications plans, um help you reviewlegal documents,all of these things that would havetaken a lot of people and a lot ofexpertise and you now have GPT5 to helpyou do all all of this. That that'spretty amaz

In [ ]:
## Retriever

In [20]:
retriever = vector_store.as_retriever(search_kwargs = {"k":4})

In [21]:
retriever.invoke("How to start a company ?")

[Document(id='98bc040f-c40c-4318-8fc3-6cdc1662e1c0', metadata={}, page_content="I thought we'll keep todayabout first principles and how the worldis changing by virtue of all that ischanging in the world that you dominate.So the very first thing I want to startwith is if I were a 25year-oldboy or girl living in Mumbai orBangalore in India. Uh I know you'vesaid a bunch of times that colleges arenotare not holding on to the place ofrelevance they might have had when I wasgrowing up. But what do I do now? A whatdo I study? If I'm starting a company,what kind of company do I"),
 Document(id='55e2613d-49cd-462f-9186-19d34a648580', metadata={}, page_content="I'm starting a company,what kind of company do I start?Or if I were to even find a job, whatindustry do you think has some kind oftailwind? I'm not talking 10 years downthe line, but even as close as three tofive years down the line.>> First of all, uh I think this isprobably the most exciting time to bestarting out one's career. Uh mayb

In [ ]:
## Augmentation

In [22]:
prompt = PromptTemplate(template="""You are a Helpful assistant . Answer from the following context . If the context is insufficient Then Directly say I dont know .
(context) Question {question}""",
input_variables=['context','question'])

In [46]:
question = "What is nikhil kamath talking about?"

In [47]:
context = retriever.invoke(question)

In [48]:
context = "".join(i.page_content for i in context)

In [26]:
context

"I thought we'll keep todayabout first principles and how the worldis changing by virtue of all that ischanging in the world that you dominate.So the very first thing I want to startwith is if I were a 25year-oldboy or girl living in Mumbai orBangalore in India. Uh I know you'vesaid a bunch of times that colleges arenotare not holding on to the place ofrelevance they might have had when I wasgrowing up. But what do I do now? A whatdo I study? If I'm starting a company,what kind of company do II'm starting a company,what kind of company do I start?Or if I were to even find a job, whatindustry do you think has some kind oftailwind? I'm not talking 10 years downthe line, but even as close as three tofive years down the line.>> First of all, uh I think this isprobably the most exciting time to bestarting out one's career. Uh maybeever. I I I think like that 25-year-oldin Mumbai can probably do more than anyprevious 25-year-old in in historycould. It's really amazing what you cando withthat

In [49]:
final_prompt = prompt.invoke({'question': question , 'context':context})

In [68]:
## Generation

In [50]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.2
)

In [51]:
answer = llm.invoke(final_prompt)

In [52]:
answer.content


"I don't know."